In [75]:
# import os
# os.environ["HF_TOKEN"]="xxxxxxxxxxxxxxxxxxx"

### Installing Libraries

In [76]:
!pip install -q youtube-transcript-api langchain-community langchain-huggingface faiss-cpu tiktoken python-dotenv

In [77]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate


### Step 1.A : Indexing (Document Ingestion)

In [99]:
# from youtube_transcript_api import YouTubeTranscriptApi
# from youtube_transcript_api._errors import TranscriptsDisabled

video_id = "XR1e6t_lSMk" #only the video id, not the entire url
try:
    fetched_transcript = YouTubeTranscriptApi().fetch(video_id, languages=['en'])
    transcript_list = fetched_transcript.to_raw_data()  #
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript[:500])

except TranscriptsDisabled:
    print("Transcripts are disabled for this video.")



(upbeat music) As a young kid, I had big dreams of becoming a concert violinist and really poured every ounce of my being into this pursuit from the time that I was six years old. I saw that as my past present and future, and I think it held me back. I would've benefited a lot from having a more malleable sense of self. What is my version of my through line? Like if you strip away all the superficial features of a pursuit what's left, that really ignites passion in me. If there is a central them


In [115]:
print(fetched_transcript[:10])

[FetchedTranscriptSnippet(text='(upbeat music)', start=0.282, duration=2.878), FetchedTranscriptSnippet(text='As a young kid, I had big dreams of becoming', start=3.16, duration=2.534), FetchedTranscriptSnippet(text='a concert violinist and really poured every ounce', start=5.694, duration=4.126), FetchedTranscriptSnippet(text='of my being into this pursuit from the time', start=9.82, duration=2.69), FetchedTranscriptSnippet(text='that I was six years old.', start=12.51, duration=1.25), FetchedTranscriptSnippet(text='I saw that as my past present and future,', start=15.36, duration=3.017), FetchedTranscriptSnippet(text='and I think it held me back.', start=18.377, duration=3.093), FetchedTranscriptSnippet(text="I would've benefited a lot from", start=21.47, duration=2.089), FetchedTranscriptSnippet(text='having a more malleable sense of self.', start=23.559, duration=3.454), FetchedTranscriptSnippet(text='What is my version of my through line?', start=28.23, duration=2.09)]


In [114]:
print(transcript_list[:10])

[{'text': '(upbeat music)', 'start': 0.282, 'duration': 2.878}, {'text': 'As a young kid, I had big dreams of becoming', 'start': 3.16, 'duration': 2.534}, {'text': 'a concert violinist and really poured every ounce', 'start': 5.694, 'duration': 4.126}, {'text': 'of my being into this pursuit from the time', 'start': 9.82, 'duration': 2.69}, {'text': 'that I was six years old.', 'start': 12.51, 'duration': 1.25}, {'text': 'I saw that as my past present and future,', 'start': 15.36, 'duration': 3.017}, {'text': 'and I think it held me back.', 'start': 18.377, 'duration': 3.093}, {'text': "I would've benefited a lot from", 'start': 21.47, 'duration': 2.089}, {'text': 'having a more malleable sense of self.', 'start': 23.559, 'duration': 3.454}, {'text': 'What is my version of my through line?', 'start': 28.23, 'duration': 2.09}]


### Step 1.B : Indexing (Text Splitting)

In [81]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.create_documents([transcript])

In [82]:
len(chunks)

368

In [83]:
chunks[0]

Document(metadata={}, page_content="(upbeat music) As a young kid, I had big dreams of becoming a concert violinist and really poured every ounce of my being into this pursuit from the time that I was six years old. I saw that as my past present and future, and I think it held me back. I would've benefited a lot from having a more malleable sense of self. What is my version of my through line? Like if you strip away all the superficial features of a pursuit what's left, that really ignites passion in me. If there is a central")

In [84]:
chunks[1]

Document(metadata={}, page_content="features of a pursuit what's left, that really ignites passion in me. If there is a central theme to this podcast, and I think there is, it is transformation. In other words, the mechanics of growth, how to change and what to do when life changes around us. Well, this subject matter also just so happens to be the core expertise of today's guest, the delightful and highly esteem. Dr. Maya Shankar Dr. Shankar is a cognitive neuroscientist, as well as the host of the podcast, A Slight Change of")

### Step 1.C & 1.D : Indexing (Embedding Creation & Vector Store)

In [ ]:
!pip install --upgrade pillow sentence-transformers transformers faiss-cpu

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

In [102]:
list(vectorstore.index_to_docstore_id.items())[:10]

[(0, '0f009bac-436f-4c1b-a3cf-1f32d6d5c083'),
 (1, '4db57d73-052c-41fc-bcf8-892500c7814e'),
 (2, 'ff36a507-4d5c-46a9-9ae1-784caf3ca62d'),
 (3, '446cd369-28fe-4aef-8e3b-fda72a9afeb6'),
 (4, 'fdeeaf7c-45b1-4b9e-8b1b-5691573cc399'),
 (5, 'baa8e840-76f5-431d-8751-348994beb379'),
 (6, '22f7e22d-80a0-466f-bd6c-02e4018d2fa8'),
 (7, 'ad83a9db-fc70-4b08-b727-3da13774d499'),
 (8, '605acb27-6b2a-466c-b375-f1805a110bd8'),
 (9, 'ad632916-f313-4dd4-8ece-6a711ef593b3')]

In [103]:
vectorstore.get_by_ids(['fdeeaf7c-45b1-4b9e-8b1b-5691573cc399'])

[Document(id='fdeeaf7c-45b1-4b9e-8b1b-5691573cc399', metadata={}, page_content="just chew on that. In any event, today we discuss her path from violinist to the white house, through a behavioral science take and the nature of change. We talk about the power of something called nudges, as well as the importance of transparency. We discuss something called identity foreclosure. And we talk about the downside of present mindedness as well as many other very interesting topics. As you will soon see, Maya's just absolutely. I think you're gonna love this one, so please make")]

### Step 2 : Retrieval

In [104]:
retreiver = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":4})

In [105]:
retreiver

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000028C3699AD40>, search_kwargs={'k': 4})

In [106]:
retreiver.invoke('What is this podcast about?')

[Document(id='f310a541-1f66-4b5c-9861-32776d4463cb', metadata={}, page_content="populated by scores of beautiful and brilliant people who have amazing stories to share. Those that we don't know who can teach us something new and leave us all the better for the experience of their sharing. And so I've dedicated my career to tracking down the most compelling profits on the planet, going deep with each of them on my podcast to elucidate the best of what they have to offer and to sharing the insights gleaned for the benefit of all. But the podcast is not the only medium by"),
 Document(id='980a1b4d-ad7e-486e-9248-ab3c275326bc', metadata={}, page_content="but maybe that's part of it. It's like this whole thing just was so organically generated. I never had dreams of being a podcaster. I never thought that I would ever have a podcast. And it was really just passion from day in and day out. And I think that's in part with people, see in the show, which is, it is a host who's kind of on a pers

### Step 3 : Augmentation

In [107]:
prompt = PromptTemplate(
    template="""Based on the context below, answer the question directly and concisely, And say that "You dont know" if question is out of context .

Context: {context}

Question: {question}

Answer:""",
    input_variables=["context", "question"]
)


In [108]:
question = "What is this podcast about?"
retreived_docs = retreiver.invoke(question)

In [109]:
retreived_docs

[Document(id='f310a541-1f66-4b5c-9861-32776d4463cb', metadata={}, page_content="populated by scores of beautiful and brilliant people who have amazing stories to share. Those that we don't know who can teach us something new and leave us all the better for the experience of their sharing. And so I've dedicated my career to tracking down the most compelling profits on the planet, going deep with each of them on my podcast to elucidate the best of what they have to offer and to sharing the insights gleaned for the benefit of all. But the podcast is not the only medium by"),
 Document(id='980a1b4d-ad7e-486e-9248-ab3c275326bc', metadata={}, page_content="but maybe that's part of it. It's like this whole thing just was so organically generated. I never had dreams of being a podcaster. I never thought that I would ever have a podcast. And it was really just passion from day in and day out. And I think that's in part with people, see in the show, which is, it is a host who's kind of on a pers

In [110]:
context_text = "\n\n".join(doc.page_content for doc in retreived_docs)
context_text

"populated by scores of beautiful and brilliant people who have amazing stories to share. Those that we don't know who can teach us something new and leave us all the better for the experience of their sharing. And so I've dedicated my career to tracking down the most compelling profits on the planet, going deep with each of them on my podcast to elucidate the best of what they have to offer and to sharing the insights gleaned for the benefit of all. But the podcast is not the only medium by\n\nbut maybe that's part of it. It's like this whole thing just was so organically generated. I never had dreams of being a podcaster. I never thought that I would ever have a podcast. And it was really just passion from day in and day out. And I think that's in part with people, see in the show, which is, it is a host who's kind of on a personal expedition to try to understand herself and the world better. Sure. And it just, it's all come from from just a very personal place. You know, starting\n\

In [111]:
final_prompt = prompt.format(context=context_text, question=question)
final_prompt

'Based on the context below, answer the question directly and concisely, And say that "You dont know" if question is out of context .\n\nContext: populated by scores of beautiful and brilliant people who have amazing stories to share. Those that we don\'t know who can teach us something new and leave us all the better for the experience of their sharing. And so I\'ve dedicated my career to tracking down the most compelling profits on the planet, going deep with each of them on my podcast to elucidate the best of what they have to offer and to sharing the insights gleaned for the benefit of all. But the podcast is not the only medium by\n\nbut maybe that\'s part of it. It\'s like this whole thing just was so organically generated. I never had dreams of being a podcaster. I never thought that I would ever have a podcast. And it was really just passion from day in and day out. And I think that\'s in part with people, see in the show, which is, it is a host who\'s kind of on a personal exp

### Step 4: Generation

In [ ]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Using distilgpt2 (small, fast)
model_id = "distilgpt2"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.2,
    do_sample=True,
    device=-1  # CPU
)

llm = HuggingFacePipeline(pipeline=pipe)

In [113]:
answer = llm.invoke(final_prompt)
answer_only = answer.split("Question: What is this podcast about?")[-1].strip()
print("Question: ", question)
print( answer_only)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question:  What is this podcast about?
Answer: This isn't an interview or anything else! There are some things there (like) where every person has been asked questions before; here at The Podcasts website, one-on--you'll find answers through interviews throughout our entire series - including how many times did someone ask me when he said 'I'm gonna do whatever.' That means even though everyone knows exactly why somebody wants to be interviewed...and then after those conversations go back over time again...we also want to hear whether anyone will actually listen during any given period -- especially since these days nobody does anymore than myself....So yeah, yes, sometimes other guys might take notes too long without asking themselves quite honestly..But nowadays i am always thinking ahead along the lines of saying nothing unless absolutely necessary :)
